# Lab 27-02 · Inside a Phone SoC

See the [lab handout](README.md). Each part asks you to *predict* before you run. All parameters are assumed for teaching.

In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import socmodel as s
big = dict(ipc=6.0, c=1.2, v0=0.45, k=0.12, ps=0.10, f_max=4.2)
little = dict(ipc=2.5, c=0.22, v0=0.45, k=0.30, ps=0.02, f_max=2.4)

## Part 1 · Moving the crossover

*Predict:* the crossover if the little core's slope falls to 0.25 V/GHz.

In [2]:
better = dict(little, k=0.25)
print(f"crossover: slope 0.30 -> {s.crossover_gips(big, little):.2f} GIPS; slope 0.25 -> {s.crossover_gips(big, better):.2f} GIPS")
perf = np.linspace(0.3, 6.0, 200)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(perf, [s.epi_at_perf(p, big) for p in perf], label="big")
ax.plot(perf, [s.epi_at_perf(p, little) for p in perf], label="little, 0.30 V/GHz")
ax.plot(perf, [s.epi_at_perf(p, better) for p in perf], "--", label="little, 0.25 V/GHz")
ax.set_xlabel("performance (GIPS)"); ax.set_ylabel("energy per instruction (nJ)"); ax.set_ylim(0, 0.25); ax.legend()
plt.show()

crossover: slope 0.30 -> 4.00 GIPS; slope 0.25 -> 4.81 GIPS


## Part 2 · Leakage and the best frequency

*Predict:* how the big core's minimum-energy frequency moves as leakage rises from 0.05 to 1.0 W.

In [3]:
leak = np.linspace(0.05, 1.0, 40)
fstar = [s.f_min_energy(dict(big, ps=p)) for p in leak]
print(f"f* at 0.05 W: {fstar[0]:.3f} GHz; at 1.0 W: {fstar[-1]:.2f} GHz")
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(leak, fstar)
ax.set_xlabel("leakage power (W)"); ax.set_ylabel("minimum-energy frequency (GHz)")
plt.show()

f* at 0.05 W: 0.578 GHz; at 1.0 W: 2.20 GHz


## Part 3 · LPDDR6 and on-device models

*Predict:* the decode bound for the 3-billion-parameter, 2-bit model on LPDDR6-14400.

In [4]:
bw5, bw6 = s.lpddr_bandwidth_gbs(10667), s.lpddr_bandwidth_gbs(14400)
print(f"LPDDR5X-10667 {bw5:.1f} GB/s, LPDDR6-14400 {bw6:.1f} GB/s; 2-bit 3B model: {s.decode_tokens_per_s(bw6, 3, 2):.1f} tokens/s")
bits = np.array([2, 3, 4, 6, 8, 16])
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(bits, [s.decode_tokens_per_s(bw5, 3, b) for b in bits], "o-", label="LPDDR5X-10667")
ax.plot(bits, [s.decode_tokens_per_s(bw6, 3, b) for b in bits], "s-", label="LPDDR6-14400")
ax.set_xlabel("bits per weight"); ax.set_ylabel("decode bound (tokens/s)"); ax.legend()
plt.show()

LPDDR5X-10667 85.3 GB/s, LPDDR6-14400 115.2 GB/s; 2-bit 3B model: 153.6 tokens/s


## Part 4 · Die cost by node

*Predict:* the cost of a 120 mm² die at 3 nm and 2 nm.

In [5]:
for name, w, d0 in (("3 nm", 18500, 0.10), ("2 nm", 30000, 0.15)):
    dpw, y, c = s.die_cost(w, 120, d0)
    print(f"{name}: {dpw} dies, yield {y:.3f}, ${c:.2f} per good die")
areas = np.linspace(60, 200, 50)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(areas, [s.die_cost(18500, a, 0.10)[2] for a in areas], label="3 nm")
ax.plot(areas, [s.die_cost(30000, a, 0.15)[2] for a in areas], label="2 nm")
ax.set_xlabel("die area (mm²)"); ax.set_ylabel("cost per good die ($)"); ax.legend()
plt.show()

3 nm: 528 dies, yield 0.887, $39.51 per good die
2 nm: 528 dies, yield 0.835, $68.02 per good die
